# AlphaFold3 Structure Prediction

This notebook demonstrates multi-modal structure prediction using AlphaFold3 from Google DeepMind. Unlike its predecessor, AlphaFold3 jointly predicts the structures of proteins, nucleic acids (DNA and RNA), ligands, and their complexes using a diffusion-based architecture that generates multiple structural samples per seed. We demonstrate `run_alphafold3` by predicting the structure of the MfnG protein bound to L-tyrosine, a practical example of protein-ligand complex prediction where both protein fold and small molecule geometry must be resolved simultaneously.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphafold3")
display_overview("alphafold3")
display_docs_section("alphafold3", "Background")

# AlphaFold3

> AlphaFold3 is the latest generation structure prediction model from Google DeepMind that predicts 3D structures of biomolecular complexes including proteins, DNA, RNA, ligands, ions, and chemical modifications. It uses a [diffusion](https://en.wikipedia.org/wiki/Diffusion_model)-based architecture to generate joint 3D structures revealing how molecules fit together.

## Background

**What does this tool measure/predict?**
AlphaFold3 predicts the 3D atomic coordinates of biomolecular complexes from sequences. It outputs full-atom structures with comprehensive confidence scores including per-atom pLDDT, predicted aligned error (PAE), pTM, ipTM, and a composite ranking score. Unlike its predecessors, AlphaFold3 generates predictions for proteins alongside DNA, RNA, small molecule ligands, ions, and post-translational modifications.

**Why is this important?**
Understanding how biomolecules interact is fundamental to biology and drug discovery:

* Drug design: Predicting how small molecule drugs bind to protein targets
* Gene regulation: Modeling protein-DNA interactions that control gene expression
* RNA biology: Understanding protein-RNA complexes involved in cellular processes
* Enzyme mechanisms: Visualizing cofactor and ion binding sites
* Glycobiology: Modeling glycan modifications on proteins
* Antibody engineering: Predicting antibody-antigen binding interfaces

**Scientific foundation:**
AlphaFold3 uses a next-generation architecture combining:

1. **Pairformer module**: An improved version of the Evoformer architecture from AlphaFold2, using triangular attention to process sequence and structural features across all molecule types.
2. **Diffusion-based structure generation**: Starting from random atomic coordinates, a diffusion network iteratively refines positions to generate physically realistic 3D structures;similar to AI image generators but for molecular structures.
3. **[Multiple sequence alignments](https://en.wikipedia.org/wiki/Multiple_sequence_alignment) (MSAs)**: Evolutionary information from homologous sequences improves prediction accuracy for proteins and RNA.

Confidence metrics include:

* **pLDDT** (predicted Local Distance Difference Test): Per-atom confidence score (0-100), where >90 indicates high confidence, 70-90 is moderate, and \<50 suggests the region is probably wrong.
* **pTM** (predicted Template Modeling score): Overall structure accuracy (0-1), where >0.5 indicates the global fold might be correct.
* **ipTM** (interface pTM): Confidence in inter-chain interfaces (0-1), where >0.8 indicates high-confidence interactions and \<0.6 suggests likely failed prediction.
* **PAE** (Predicted Aligned Error): Estimates error in relative positions between residue pairs; lower values indicate higher confidence.
* **ranking\_score**: Composite metric combining ipTM, pTM, disorder penalty, and clash penalty for ranking multiple predictions.

## Available tools

In [2]:
display_available_tools("alphafold3")

- **`run_alphafold3()`** — Protein structure prediction using AlphaFold3

### `run_alphafold3`

AlphaFold3 extends structure prediction beyond proteins to jointly model proteins, DNA, RNA, ligands, and their complexes in a single unified framework. Its diffusion-based architecture generates five structural samples per seed and automatically selects the highest-confidence prediction. SMILES strings for small molecules are automatically converted to CCD (Chemical Component Dictionary) codes internally. The `seeds` parameter controls stochastic sampling and can be extended to multiple values for more thorough exploration of conformational space, which is particularly useful for challenging tasks such as antibody-antigen docking.

In [3]:
from pathlib import Path

from proto_tools import (
    AlphaFold3Config,
    AlphaFold3Input,
    Chain,
    StructurePredictionComplex,
    run_alphafold3,
)

In [4]:
# Display input docs
display_api_reference("alphafold3", "input", "run_alphafold3")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = AlphaFold3Input(complexes=[complex])

**Input** — `AlphaFold3Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | required | List of complexes to predict structures for. Inherited from `StructurePredictionInput`. Each complex can contain one or more sequences of proteins, DNA, RNA, or ligands. |
| `chains` | `List[Chain]` | required | Chains in the complex. Each chain can be: |
| `msas` | `Dict[string, MSA]` |  | Pre-computed MSAs keyed by protein sequence. Populated by preprocess() or supplied directly. Default: None. |

In [5]:
# Display config docs
display_api_reference("alphafold3", "config", "run_alphafold3")

# Configure AlphaFold3
config = AlphaFold3Config(
    name="mfng_ligand",
    seeds=[0],  # AlphaFold3 generates 5 diffusion samples per seed
    use_msa=False,
    verbose=False,
)

**Config** — `AlphaFold3Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `name` | `string` | `af3_job` | Name of the folding job. Default: `"af3_job"`. |
| `seeds` | `List[integer]` | `[0]` | Seeds to use for AlphaFold3. Default: `[0]`. |
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `use_msa` | `boolean` | `True` | Whether to generate and use Multiple Sequence Alignments (MSAs) for protein chains using ColabFold search. Inherited from `MSAStructurePredictionConfig`. Default: `True`. |
| `output_dir` | `string` |  | Path prefix for the AlphaFold3 output directory. Appends `_af3_results` to the provided string. If `None` (default), uses a temporary directory that is automatically cleaned up after inference. If specified, creates a persistent directory at the given path that will NOT be automatically deleted. Default: `None`. |
| `repo_path` | `string` | `/large_storage/hielab/brk/models/alphafold3` | Local path to the cloned AlphaFold3 repository. Required for execution. |
| `sif_path` | `string` | `/large_storage/hielab/brk/models/alphafold3/alphafold3_latest.sif` | Local path to the AlphaFold3 Singularity image file (.sif). Required for container execution. |
| `model_dir` | `string` | `/large_storage/hielab/brk/models/af3_weights` | Local path to the directory containing AlphaFold3 model parameters/weights. |
| `db_dir` | `string` | `/large_storage/hielab/brk/databases/af3_dbs` | Local path to the AlphaFold3 genetic databases. |
| `verbose` | `boolean` | `False` | Whether to print status messages during execution. Inherited from `StructurePredictionConfig`. Default: `False`. |
| `device` | `string` | `cuda` | Device to run the model on (`"cuda"`, `"cpu"`). Inherited from `StructurePredictionConfig`. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |
| `colabfold_search_config` | `ColabfoldSearchConfig` |  | Configuration for ColabFold MSA search. Only used when `use_msa=True`. Inherited from `MSAStructurePredictionConfig`. Default: `None`. |

In [6]:
# Run structure prediction
result = run_alphafold3(inputs, config)

Removing stale standalone helpers at /large_storage/hielab/bviggiano/codebases/evo-design/proto-tools/proto_tools/tools/structure_scoring/structure_metrics/standalone (tool was likely moved to a different category)


Folding structures (AlphaFold3):   0%|          | 0/1 [00:00<?, ?complex/s]

Setup files changed for alphafold3, rebuilding environment


Removing environment at /large_storage/hielab/bviggiano/proto_cache/proto_tool_envs/alphafold3_env


In [7]:
# Display output docs
display_api_reference("alphafold3", "output", "run_alphafold3")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Ranking score: {mfng_structure.metrics.ranking_score:.3f}")
print(f"  Average pLDDT: {mfng_structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.metrics.iptm:.3f}")

**Output** — `AlphaFold3Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `List[Structure]` | required | Predicted structures, each carrying an :class:`AlphaFold3Metrics` instance on `.metrics`. |
| `structure` | `string` | required | Raw structure content in PDB or CIF format. |
| `structure_format` | `string` |  | Format of the content string (auto-detected if omitted). |
| `b_factor_type` | `BFactorType` |  | What the B-factor column represents. |
| `source` | `string` |  | Optional source identifier (filepath or tool name). |
| `metrics` | `Metrics` |  | Associated metrics (e.g., pLDDT, pTM scores, per-chain lists, pairwise matrices). None values are stripped at construction. |

  Number of chains: 2
  Protein length: 384 residues
  Ranking score: 0.490
  Average pLDDT: 36.665
  pTM score: 0.310
  ipTM score: 0.520


#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain. The protein backbone is shown alongside the bound L-tyrosine ligand, allowing you to inspect the predicted binding pose and overall fold quality.

In [8]:
mfng_structure.visualize(style='stick', color_by='chain')

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT scores for per-residue confidence visualization.

In [9]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")

Structure exported to example_output/mfng_ligand_complex.pdb
